In [10]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
INTERIMDIR = CONFIGS['filepaths']['interim']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SPLIT      = 'test'

MODELS = {}
for name,rc in CONFIGS['experiments']['pod']['runs'].items():
    MODELS[name] = rc['description']
for name,rc in CONFIGS['experiments']['nn']['runs'].items():
    MODELS[name] = rc['description']
for name,rc in CONFIGS['experiments']['sr']['optimizedeqs'].items():
    MODELS[name] = rc['description']

In [ ]:
def spatial_r2(ytrue,ypred):
    ssres = ((ytrue-ypred)**2).sum('time',skipna=True)
    sstot = ((ytrue-ytrue.mean('time',skipna=True))**2).sum('time',skipna=True)
    return (1-ssres/sstot).squeeze()

def scalar_r2(ytrue,ypred,land_mask):
    ssres = ((ytrue-ypred)**2).sum('time',skipna=True).values.ravel()
    sstot = ((ytrue-ytrue.mean('time',skipna=True))**2).sum('time',skipna=True).values.ravel()
    fin   = np.isfinite(ssres)&np.isfinite(sstot)
    land  = land_mask.ravel()
    r2l   = float(1-ssres[land &fin].sum()/sstot[land &fin].sum())
    r2o   = float(1-ssres[~land&fin].sum()/sstot[~land&fin].sum())
    return r2l,r2o

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

results = {}
for name in MODELS:
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):continue
    with xr.open_dataset(filepath) as ds:
        pred = ds.tp.load()
    if 'seed' in pred.dims:pred = pred.mean('seed')
    if 'complexity' in pred.dims:pred = pred.isel(complexity=0)
    ytrue,ypred   = xr.align(truetp,pred,join='inner')
    results[name] = (ytrue.squeeze(),ypred.squeeze())

print(f'Loaded {len(results)}/{len(MODELS)} models!')

with xr.open_dataset(os.path.join(INTERIMDIR,'lf.nc')) as ds:
    lf = ds[list(ds.data_vars)[0]].squeeze().load()
ref  = next(iter(results.values()))[0].isel(time=0)
lf   = lf.interp(lat=ref.lat,lon=ref.lon,method='nearest')
LAND = (lf.values>=0.5)
print(f'Land mask: {LAND.sum()} land, {(~LAND).sum()} ocean pixels')

In [ ]:
names = sorted([n for n in MODELS if n in results],
               key=lambda n:float(spatial_r2(*results[n]).mean()))
if 'pod_bl' in names and 'sr_full' in names:
    i_pod,i_sr = names.index('pod_bl'),names.index('sr_full')
    if i_sr<i_pod:
        names.remove('sr_full')
        names.insert(names.index('pod_bl')+1,'sr_full')
n     = len(names)
ncols = 3
nrows = int(np.ceil(n/ncols))

fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',figwidth=6.5,share=True)
axs.format(coast=True,borders=False,latlim=LATRANGE,lonlim=LONRANGE,latlines=5,lonlines=5,grid=False)
m = None
for ax,name in zip(axs,names):
    ytrue,ypred = results[name]
    r2          = spatial_r2(ytrue,ypred)
    r2l,r2o     = scalar_r2(ytrue,ypred,LAND)
    m = ax.pcolormesh(r2.lon,r2.lat,r2,cmap='ColdHot',cmap_kw={'left':0.5},vmin=0,vmax=1,levels=21)
    ax.format(title=MODELS.get(name,name))
    ax.text(0.02,0.03,f'Land R$^2$ = {r2l:.2f} \nOcean R$^2$ = {r2o:.2f}',
            transform=ax.transAxes,ha='left',va='bottom',
            bbox=dict(facecolor='white',alpha=0.5,edgecolor='none',pad=2))
for ax in axs[n:]:
    ax.set_visible(False)
fig.colorbar(m,loc='b',label='Time-Mean R$^2$',ticks=0.1)
pplt.show()
fig.save('../figs/fig_3.jpg')